# Práctica · Semana 2: Búsqueda no informada en Python

**Curso:** Introducción a la Inteligencia Artificial · UDES
**Tiempo estimado:** 2 horas
**Relación con el capítulo:** secciones 2.3 a 2.8

En este cuaderno vas a ejecutar, modificar y comparar los algoritmos de búsqueda no informada del capítulo 2 sobre los tres problemas que usa García Serrano (2016, caps. 2 y 3): el **puzle lineal**, los **vuelos con el mínimo número de trasbordos** y las **rutas por carretera con la menor distancia**.

El código es una adaptación a Python 3 de los listados 3-2 a 3-13 de García Serrano (2016), que están escritos en Python 2. Los cambios principales son:

- `print` como función y `collections.deque` como cola FIFO.
- Conjuntos (`set`) para comprobar en tiempo constante si un estado ya fue visitado.
- `heapq` como cola con prioridad en la búsqueda de coste uniforme (el libro ordena la lista con `sorted(..., cmp=...)`, que ya no existe en Python 3).
- Un contador de nodos expandidos para comparar el esfuerzo de cada algoritmo.

**Cómo trabajar:** ejecuta las celdas en orden (`Shift + Enter`). Las celdas marcadas con ✏️ son ejercicios: escribe tus respuestas en las celdas de texto que siguen a cada una.

> Para abrirlo en Google Colab: *Archivo → Subir cuaderno* y selecciona este archivo, o ábrelo desde GitHub con *Archivo → Abrir cuaderno → GitHub*.

## 1. La estructura de datos `Nodo`

Un **nodo** del árbol de búsqueda guarda el estado, el nodo padre, la acción que lo generó, el coste del camino $g(n)$ y la profundidad. Son los cinco campos que describen Russell y Norvig (2004, p. 78). García Serrano (2016, listado 3-2) usa una clase muy parecida.

El método `camino()` sigue los punteros al padre desde el nodo objetivo hasta la raíz para reconstruir la solución.

In [ ]:
from collections import deque
import heapq


class Nodo:
    # Nodo del árbol de búsqueda (adaptado de García Serrano, 2016, listado 3-2).

    def __init__(self, estado, padre=None, accion=None, coste=0):
        self.estado = estado
        self.padre = padre
        self.accion = accion
        self.coste = coste                      # g(n): coste del camino desde la raíz
        self.profundidad = 0 if padre is None else padre.profundidad + 1

    def camino(self):
        # Devuelve la lista de estados desde la raíz hasta este nodo.
        nodo, estados = self, []
        while nodo is not None:
            estados.append(nodo.estado)
            nodo = nodo.padre
        return list(reversed(estados))

    def __repr__(self):
        return f"Nodo({self.estado}, g={self.coste}, prof={self.profundidad})"

## 2. Problema 1: el puzle lineal

Cuatro piezas numeradas deben quedar en el orden `(1, 2, 3, 4)`. Hay tres operadores (García Serrano, 2016, cap. 3):

| Operador | Efecto |
|:-:|---|
| **I** | intercambia las dos piezas de la izquierda |
| **C** | intercambia las dos piezas centrales |
| **D** | intercambia las dos piezas de la derecha |

Usamos tuplas en lugar de listas porque las tuplas se pueden guardar en un conjunto (`set`).

In [ ]:
def sucesores_puzle(estado):
    # Aplica los operadores I, C y D, en ese orden.
    e = estado
    return [
        ("I", (e[1], e[0], e[2], e[3])),
        ("C", (e[0], e[2], e[1], e[3])),
        ("D", (e[0], e[1], e[3], e[2])),
    ]

sucesores_puzle((3, 1, 2, 4))

### 2.1 Búsqueda en amplitud (BFS)

La frontera es una **cola FIFO**: el primer nodo que entra es el primero que se expande. El algoritmo sigue el pseudocódigo del listado 3-1 de García Serrano (2016). No añade a la frontera un hijo que ya esté en `visitados` o en la frontera.

In [ ]:
def bfs(inicial, objetivo, sucesores):
    frontera = deque([Nodo(inicial)])
    en_frontera = {inicial}
    visitados = set()
    expandidos = 0
    while frontera:
        nodo = frontera.popleft()               # FIFO: sale el más antiguo
        en_frontera.discard(nodo.estado)
        visitados.add(nodo.estado)
        if nodo.estado == objetivo:
            return nodo, expandidos
        expandidos += 1
        for accion, hijo in sucesores(nodo.estado):
            if hijo not in visitados and hijo not in en_frontera:
                frontera.append(Nodo(hijo, nodo, accion))
                en_frontera.add(hijo)
    return None, expandidos


def mostrar(resultado):
    nodo, expandidos = resultado
    if nodo is None:
        print(f"Sin solución. Nodos expandidos: {expandidos}")
        return
    ruta = nodo.camino()
    print("Camino:", " -> ".join(str(s) for s in ruta))
    print(f"Pasos: {len(ruta) - 1} | Coste g(n): {nodo.coste} | Nodos expandidos: {expandidos}")


mostrar(bfs((4, 2, 3, 1), (1, 2, 3, 4), sucesores_puzle))

El camino coincide con el que reporta García Serrano (2016, listado 3-3): cinco movimientos.

### 2.2 Búsqueda en profundidad (DFS)

El único cambio es la estructura de la frontera: una **pila LIFO**, así que el último nodo que entra es el primero que se expande (García Serrano, 2016, listado 3-5).

In [ ]:
def dfs(inicial, objetivo, sucesores):
    frontera = [Nodo(inicial)]
    en_frontera = {inicial}
    visitados = set()
    expandidos = 0
    while frontera:
        nodo = frontera.pop()                   # LIFO: sale el más reciente
        en_frontera.discard(nodo.estado)
        visitados.add(nodo.estado)
        if nodo.estado == objetivo:
            return nodo, expandidos
        expandidos += 1
        for accion, hijo in sucesores(nodo.estado):
            if hijo not in visitados and hijo not in en_frontera:
                frontera.append(Nodo(hijo, nodo, accion))
                en_frontera.add(hijo)
    return None, expandidos


mostrar(dfs((4, 2, 3, 1), (1, 2, 3, 4), sucesores_puzle))

DFS encuentra una solución de 11 movimientos, la misma que muestra García Serrano (2016, listado 3-6). Es una solución válida, pero **no óptima**.

### ✏️ Ejercicio 1. Amplitud contra profundidad

Ejecuta la celda siguiente, que resuelve los 24 estados iniciales posibles del puzle con ambos algoritmos.

1. ¿En cuántos casos DFS encuentra una solución más larga que BFS? ¿Alguna vez la encuentra más corta?
2. Compara la columna de nodos expandidos caso por caso. ¿Hay estados iniciales en los que DFS expande muchos menos nodos que BFS? ¿Y al revés? ¿Qué muestran los promedios?
3. Con base en lo anterior, explica la diferencia entre **optimalidad** (la calidad de la solución) y **coste de la búsqueda** (el esfuerzo para encontrarla).

In [ ]:
from itertools import permutations

objetivo = (1, 2, 3, 4)
filas = []
for inicial in permutations(objetivo):
    nb, eb = bfs(inicial, objetivo, sucesores_puzle)
    nd, ed = dfs(inicial, objetivo, sucesores_puzle)
    filas.append((inicial, nb.profundidad, eb, nd.profundidad, ed))

print(f"{'Inicial':<14}{'BFS pasos':>10}{'BFS exp.':>10}{'DFS pasos':>11}{'DFS exp.':>10}")
for f in filas:
    print(f"{str(f[0]):<14}{f[1]:>10}{f[2]:>10}{f[3]:>11}{f[4]:>10}")

print("\nPromedio de pasos    BFS:", round(sum(f[1] for f in filas) / 24, 2),
      "| DFS:", round(sum(f[3] for f in filas) / 24, 2))
print("Promedio de expandidos BFS:", round(sum(f[2] for f in filas) / 24, 2),
      "| DFS:", round(sum(f[4] for f in filas) / 24, 2))

*Tus respuestas al ejercicio 1:*

## 3. Problema 2: vuelos con el mínimo número de trasbordos

El grafo de conexiones es el del listado 3-4 de García Serrano (2016). Está representado con un **diccionario**: cada ciudad apunta al conjunto de ciudades a las que hay vuelo directo. Observa que las conexiones tienen **sentido**: que exista `A -> B` no garantiza que exista `B -> A`.

Usamos listas ordenadas alfabéticamente para que el orden de expansión, y por tanto el resultado, sea siempre el mismo.

In [ ]:
vuelos = {
    "Malaga":    ["Barcelona", "Madrid", "Salamanca"],
    "Sevilla":   ["Madrid", "Santiago"],
    "Granada":   ["Valencia"],
    "Valencia":  ["Barcelona"],
    "Madrid":    ["Barcelona", "Malaga", "Salamanca", "Santander", "Sevilla"],
    "Salamanca": ["Madrid", "Malaga"],
    "Santiago":  ["Barcelona", "Santander", "Sevilla"],
    "Santander": ["Madrid", "Santiago"],
    "Zaragoza":  ["Barcelona"],
    "Barcelona": ["Madrid", "Malaga", "Santiago", "Valencia", "Zaragoza"],
}

def sucesores_vuelos(ciudad):
    return [(f"volar a {d}", d) for d in vuelos.get(ciudad, [])]

mostrar(bfs("Malaga", "Santiago", sucesores_vuelos))

BFS encuentra `Malaga -> Barcelona -> Santiago`, la misma respuesta de García Serrano (2016, cap. 3): no hay ninguna ruta con menos trasbordos.

### 3.1 Búsqueda en profundidad limitada y profundidad iterativa

La **búsqueda en profundidad limitada** es una DFS recursiva que no desciende más allá de un límite. Puede terminar de dos maneras distintas: `FALLO` (no hay solución) o `CORTE` (no hay solución *dentro del límite*) (Russell y Norvig, 2004, p. 87, fig. 3.13).

La **búsqueda con profundidad iterativa** repite la búsqueda limitada con límites 0, 1, 2, … hasta encontrar el objetivo (García Serrano, 2016, listados 3-9 y 3-10; Russell y Norvig, 2004, pp. 87-88).

Siguiendo a Russell y Norvig, aquí solo se evitan los ciclos del camino actual (un estado no se repite dentro de la misma rama), lo que mantiene la memoria proporcional a la profundidad.

In [ ]:
CORTE, FALLO = "corte", "fallo"

def dfs_limitada(nodo, objetivo, sucesores, limite, contador):
    if nodo.estado == objetivo:
        return nodo
    if nodo.profundidad == limite:
        return CORTE
    contador[0] += 1                            # nodo expandido
    hubo_corte = False
    en_camino = set(nodo.camino())              # evita ciclos en la rama actual
    for accion, hijo in sucesores(nodo.estado):
        if hijo in en_camino:
            continue
        resultado = dfs_limitada(Nodo(hijo, nodo, accion), objetivo, sucesores, limite, contador)
        if resultado == CORTE:
            hubo_corte = True
        elif resultado != FALLO:
            return resultado
    return CORTE if hubo_corte else FALLO


def profundidad_iterativa(inicial, objetivo, sucesores, limite_max=50, verbose=True):
    total = [0]
    for limite in range(limite_max + 1):
        contador = [0]
        resultado = dfs_limitada(Nodo(inicial), objetivo, sucesores, limite, contador)
        total[0] += contador[0]
        if verbose:
            print(f"  límite {limite}: {contador[0]} nodos expandidos -> "
                  f"{resultado if isinstance(resultado, str) else 'solución'}")
        if resultado != CORTE:
            return (None if resultado == FALLO else resultado), total[0]
    return None, total[0]


mostrar(profundidad_iterativa("Malaga", "Santiago", sucesores_vuelos))

La profundidad iterativa llega a la misma solución óptima que BFS, como indica García Serrano (2016, listado 3-11).

### ✏️ Ejercicio 2. Una ruta que el capítulo 2 sí encuentra

García Serrano (2016, cap. 2) afirma que para volar de Málaga a Granada hay que visitar cuatro aeropuertos: `Málaga > Barcelona > Valencia > Granada`.

1. Ejecuta la celda siguiente. ¿Qué devuelven BFS y la profundidad iterativa?
2. Revisa el diccionario `vuelos`. ¿Por qué el programa no encuentra la ruta que describe el libro? Pista: mira qué conexiones salen de `Valencia` y de `Granada`.
3. Corrige el diccionario para que las conexiones sean de ida y vuelta (un grafo no dirigido) y vuelve a ejecutar. Escribe la ruta que obtienes.

In [ ]:
print("BFS:")
mostrar(bfs("Malaga", "Granada", sucesores_vuelos))
print("\nProfundidad iterativa:")
mostrar(profundidad_iterativa("Malaga", "Granada", sucesores_vuelos, limite_max=10))

*Tus respuestas al ejercicio 2:*

## 4. Problema 3: rutas por carretera con la menor distancia

Ahora cada conexión tiene un coste en kilómetros (García Serrano, 2016, listado 3-13). Contar trasbordos ya no sirve: una ruta que pasa por más ciudades puede ser más corta.

In [ ]:
carreteras = {
    "Malaga":    {"Granada": 125, "Madrid": 513},
    "Sevilla":   {"Madrid": 514},
    "Granada":   {"Malaga": 125, "Madrid": 423, "Valencia": 491},
    "Valencia":  {"Granada": 491, "Madrid": 356, "Zaragoza": 309, "Barcelona": 346},
    "Madrid":    {"Salamanca": 203, "Sevilla": 514, "Malaga": 513, "Granada": 423,
                  "Barcelona": 603, "Santander": 437, "Valencia": 356,
                  "Zaragoza": 313, "Santiago": 599},
    "Salamanca": {"Santiago": 390, "Madrid": 203},
    "Santiago":  {"Salamanca": 390, "Madrid": 599},
    "Santander": {"Madrid": 437, "Zaragoza": 394},
    "Zaragoza":  {"Barcelona": 296, "Valencia": 309, "Madrid": 313},
    "Barcelona": {"Zaragoza": 296, "Madrid": 603, "Valencia": 346},
}

def sucesores_carretera(ciudad):
    return [(f"conducir a {d}", d) for d in sorted(carreteras[ciudad])]

def coste_camino(ruta):
    return sum(carreteras[a][b] for a, b in zip(ruta, ruta[1:]))

nodo, exp = bfs("Malaga", "Santiago", sucesores_carretera)
print("BFS:", " -> ".join(nodo.camino()), "|", coste_camino(nodo.camino()), "km")

BFS devuelve la ruta con **menos ciudades**, pero no la de menos kilómetros. Por eso García Serrano (2016, cap. 3) concluye que BFS no es óptima en este problema.

### 4.1 Búsqueda de coste uniforme (UCS)

UCS expande siempre el nodo de la frontera con menor $g(n)$. La frontera es una **cola con prioridad** (García Serrano, 2016, listado 3-12). Si un estado ya está en la frontera y aparece un camino más barato, se sustituye; con `heapq` lo hacemos insertando la nueva entrada e ignorando después la entrada vieja.

Con `traza=True` se imprime la frontera en cada paso, en el mismo formato de la tabla paso a paso del libro.

In [ ]:
def ucs(inicial, objetivo, sucesores, costo, traza=False):
    contador = 0                                 # desempate estable en el heap
    frontera = [(0, contador, Nodo(inicial))]
    mejor_g = {inicial: 0}                       # mejor coste conocido en la frontera
    visitados = set()
    expandidos = 0
    paso = 0
    while frontera:
        g, _, nodo = heapq.heappop(frontera)
        if nodo.estado in visitados or g > mejor_g.get(nodo.estado, float("inf")):
            continue                             # entrada obsoleta
        paso += 1
        if traza:
            vivos = sorted((gg, n.estado) for gg, _, n in frontera
                           if n.estado not in visitados and gg == mejor_g.get(n.estado))
            print(f"Paso {paso}. Seleccionado: {nodo.estado}({g}) | resto de la frontera: "
                  + ", ".join(f"{e}({gg})" for gg, e in vivos))
        visitados.add(nodo.estado)
        if nodo.estado == objetivo:
            return nodo, expandidos
        expandidos += 1
        for accion, hijo in sucesores(nodo.estado):
            if hijo in visitados:
                continue
            g_hijo = g + costo(nodo.estado, hijo)
            if g_hijo < mejor_g.get(hijo, float("inf")):
                mejor_g[hijo] = g_hijo
                contador += 1
                heapq.heappush(frontera, (g_hijo, contador, Nodo(hijo, nodo, accion, g_hijo)))
    return None, expandidos


km = lambda a, b: carreteras[a][b]
mostrar(ucs("Malaga", "Santiago", sucesores_carretera, km, traza=True))

UCS encuentra `Malaga -> Madrid -> Salamanca -> Santiago` con 1.106 km, igual que García Serrano (2016, cap. 3). Es la ruta óptima, aunque pase por una ciudad más que la de BFS (1.112 km).

Observa que la traza tiene 10 pasos y la tabla del libro 9. En su paso 9 el libro selecciona `Santiago(1106)` aunque en la frontera está `Sevilla(1027)`, que tiene menor coste. Una búsqueda de coste uniforme estricta expande primero Sevilla, como hace este código. La solución final es la misma.

### ✏️ Ejercicio 3. Seguir la traza

1. Compara la traza anterior con la tabla paso a paso del libro. En el paso 3, `Valencia` entra a la frontera con 616 km (vía Granada); cuando se expande Madrid aparece otra vez con 868 km. ¿Qué hace el algoritmo con ese segundo camino y por qué?
2. ¿Por qué UCS no se detiene en el paso 3, cuando genera por primera vez a `Santiago` con 1.112 km? Relaciona tu respuesta con la **prueba de objetivo al expandir** (Russell y Norvig, 2004, p. 84).
3. Calcula a mano con `coste_camino` el coste de otra ruta a Santiago que pase por Valencia. ¿Es mejor o peor?

In [ ]:
# Espacio para el punto 3: escribe una ruta y calcula su coste
coste_camino(["Malaga", "Granada", "Madrid", "Santiago"])

*Tus respuestas al ejercicio 3:*

### ✏️ Ejercicio 4. Si todos los costes son iguales

Russell y Norvig (2004, p. 84) señalan que UCS equivale a BFS cuando todos los costes individuales son iguales.

1. Ejecuta UCS sobre el problema de los vuelos usando un coste de 1 por trasbordo (celda siguiente). ¿Obtienes la misma solución que con BFS?
2. ¿Qué representa $g(n)$ en ese caso?

In [ ]:
un_trasbordo = lambda a, b: 1
mostrar(ucs("Malaga", "Santiago", sucesores_vuelos, un_trasbordo))

*Tus respuestas al ejercicio 4:*

## 5. La explosión combinatoria

Con 4 piezas el puzle tiene $4! = 24$ estados (García Serrano, 2016, cap. 3). El libro advierte que con más piezas el problema se vuelve intratable. Vamos a comprobarlo generalizando el puzle a $n$ piezas, con $n - 1$ operadores (un intercambio por cada par de piezas contiguas).

### ✏️ Ejercicio 5. ¿Hasta dónde llegamos?

1. Ejecuta la celda. Toma como estado inicial el orden invertido, que es el más alejado del objetivo.
2. Anota cómo crecen los estados posibles, los nodos expandidos y el tiempo de BFS y de profundidad iterativa.
3. Si cada nodo ocupara 1 KB (supuesto de García Serrano, 2016, cap. 3), ¿cuánta memoria necesitaría BFS para almacenar todos los estados visitados con $n = 10$?
4. Con estos datos, explica con tus palabras por qué Russell y Norvig (2004, p. 84) concluyen que los problemas de búsqueda de complejidad exponencial no se pueden resolver por métodos sin información, salvo casos pequeños.

> ⚠️ La profundidad iterativa se ejecuta solo hasta $n = 5$. Esta versión solo evita ciclos dentro de la rama actual, así que en el puzle vuelve a visitar el mismo estado por muchos caminos distintos: con $n = 6$ expande cerca de 100 millones de nodos y tarda varios minutos. Es un ejemplo del problema de los estados repetidos (Russell y Norvig, 2004, pp. 91-92).

In [ ]:
import time
from math import factorial

def sucesores_puzle_n(estado):
    res = []
    for i in range(len(estado) - 1):
        e = list(estado)
        e[i], e[i + 1] = e[i + 1], e[i]
        res.append((f"swap{i}", tuple(e)))
    return res

print(f"{'n':>2}{'estados n!':>12}{'pasos':>7}{'BFS exp.':>10}{'BFS s':>8}{'BPI exp.':>12}{'BPI s':>8}")
for n in range(3, 9):
    objetivo = tuple(range(1, n + 1))
    inicial = tuple(reversed(objetivo))
    t0 = time.perf_counter()
    nodo, exp_bfs = bfs(inicial, objetivo, sucesores_puzle_n)
    t_bfs = time.perf_counter() - t0
    if n <= 5:
        t0 = time.perf_counter()
        _, exp_bpi = profundidad_iterativa(inicial, objetivo, sucesores_puzle_n, verbose=False)
        t_bpi = f"{time.perf_counter() - t0:.2f}"
    else:
        exp_bpi, t_bpi = "-", "-"
    print(f"{n:>2}{factorial(n):>12}{nodo.profundidad:>7}{exp_bfs:>10}{t_bfs:>8.2f}{exp_bpi:>12}{t_bpi:>8}")

*Tus respuestas al ejercicio 5:*

## 6. Cierre

Resume en una tabla lo que observaste en esta práctica:

| Algoritmo | Estructura de la frontera | ¿Encontró la solución óptima en los tres problemas? | Ventaja principal | Desventaja principal |
|---|---|---|---|---|
| BFS | | | | |
| DFS | | | | |
| Profundidad iterativa | | | | |
| Coste uniforme | | | | |

Lleva este cuaderno a la sesión sincrónica. Lo que practicaste aquí es la base de la parte de modelado e implementación de la **Actividad Evaluativa 1**.

---

**Referencias**

García Serrano, A. (2016). *Inteligencia artificial: Fundamentos, práctica y aplicaciones* (2.ª ed.). RC Libros.

Russell, S. J., y Norvig, P. (2004). *Inteligencia artificial: Un enfoque moderno* (2.ª ed.; J. M. Corchado Rodríguez et al., Trads.). Pearson Educación.